# Solar AI — ground-PV CONDITION classifier, Phase 16-B (MobileNetV2)

Panel-crop CLASSIFICATION reformulation of the active 4-class condition model:
`0=CLEAN 1=DUSTY 2=BIRD_DROP 3=PHYSICAL_DAMAGE`. Motivated by
`docs/CONDITION_MODEL_IMAGE_LEVEL_AGREEMENT.md` (Phase 16-A.1): the existing BOX-DETECTOR
condition model's weakness is dominated by localization noise, not class recognition
(image-level agreement 0.727 vs. box-level recall 0.441, same checkpoint, zero new
inference) — evidence a classifier (no localization requirement) is worth trying.

- **Experimental research model only.** NOT the panel detector. Does NOT touch / replace
  the production aerial detector, the ground panel detector, or any existing FROZEN
  condition checkpoint (`weights/yolo_solar_condition_4class_research_v2/v3_960/v4_aug.pt`).
- Architecture: **MobileNetV2** (ImageNet-pretrained), same family as the project's existing
  production classifier `models/classifier.py` — reused deliberately for consistency, not a
  new architecture invented for this experiment.
- Dataset: `edithstark/solar-ai-ground-condition-4class-classifier-v1` (PRIVATE). Derived from
  the existing, already-audited 4-class DETECTION dataset by picking ONE dominant class per
  image from its existing box annotations (majority count, tie -> area, tie -> alphabetical)
  — no new photography, no new annotation, same leakage-clean split verbatim.

## ⚠️ PROVENANCE — grade C, UNVERIFIED (unchanged from the source detection dataset)
Personal, non-commercial research only. Do NOT redistribute / publish / use commercially.

## ⚠️ Information loss versus the detection dataset
910 of 2411 images had more than one distinct class in their original box annotations;
only the DOMINANT class was kept as this image's single label. Disclosed, not hidden — see
`provenance/condition_4class_classifier_v1_report.json` in the attached dataset.

Evaluated on the SAME held-out test split (by content) as the box detector, at the IMAGE
level, for an apples-to-apples comparison against the 0.727 image-level-agreement figure.


## 1. Environment


In [ ]:
import sys, platform, subprocess, json, os, time, hashlib, shutil
from pathlib import Path
print('Python :', platform.python_version())


### CUDA — fail closed if unavailable
Pin `torch==2.7.1` / `torchvision==0.22.1` (CUDA 12.6 wheels, Pascal-compatible) — the
pins proven on this account's prior ground-detector / condition runs.


In [ ]:
PINNED = ['torch==2.7.1', 'torchvision==0.22.1']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED], check=True)

import torch
assert torch.cuda.is_available(), 'CUDA unavailable — set Settings -> Accelerator -> GPU.'
import torchvision
ENV = dict(
    python=platform.python_version(), torch=torch.__version__,
    torchvision=torchvision.__version__, cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0),
    gpu_capability='.'.join(map(str, torch.cuda.get_device_capability(0))),
    gpu_memory_gb=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
)
print(json.dumps(ENV, indent=2))


## 2. Locate the attached Kaggle dataset + verify provenance


In [ ]:
# Recursive search (NOT a shallow iterdir) — Kaggle's actual mount depth under
# /kaggle/input varies (sometimes /kaggle/input/<slug>/..., sometimes nested one
# level deeper e.g. /kaggle/input/datasets/<slug>/...); rglob finds it regardless.
candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('train') if p.is_dir()})
assert candidates, 'No train/ directory found anywhere under /kaggle/input — attach the dataset in the sidebar.'
DATA_ROOT = None
for c in candidates:
    if (c / 'train').is_dir() and (c / 'valid').is_dir() and (c / 'test').is_dir():
        DATA_ROOT = c; break
assert DATA_ROOT is not None, f'no parent with train+valid+test among {candidates}'
print('DATA_ROOT:', DATA_ROOT)

EXPECTED_CLASSES = ['BIRD_DROP', 'CLEAN', 'DUSTY', 'PHYSICAL_DAMAGE']  # alphabetical — ImageFolder order
for split in ('train', 'valid', 'test'):
    classes_here = sorted(p.name for p in (DATA_ROOT / split).iterdir() if p.is_dir())
    assert classes_here == EXPECTED_CLASSES, f'{split}: {classes_here} != {EXPECTED_CLASSES}'
print('taxonomy OK (active 4-class, alphabetical ImageFolder order):', EXPECTED_CLASSES)

prov = DATA_ROOT / 'provenance' / 'condition_4class_classifier_v1_report.json'
assert prov.is_file(), 'missing provenance/condition_4class_classifier_v1_report.json'
report = json.loads(prov.read_text())
print('source dataset      :', report['source_dataset'])
print('label derivation    :', report['label_derivation_rule'])
print('total images        :', report['total_images_written'])
print('minority discarded  :', report['images_with_minority_annotation_discarded'])
assert report['total_images_written'] == 2411
assert report['per_split_class_counts']['train'] == {'BIRD_DROP': 509, 'CLEAN': 367, 'DUSTY': 749, 'PHYSICAL_DAMAGE': 61}
assert report['per_split_class_counts']['valid'] == {'BIRD_DROP': 144, 'CLEAN': 112, 'DUSTY': 207, 'PHYSICAL_DAMAGE': 20}
assert report['per_split_class_counts']['test'] == {'CLEAN': 55, 'DUSTY': 107, 'BIRD_DROP': 71, 'PHYSICAL_DAMAGE': 9}
print('per-split class counts match the local build exactly')


## 3. Data loaders


In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
INPUT_SIZE = 224  # matches configs/settings.yaml models.mobilenet.input_size

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_ds = datasets.ImageFolder(DATA_ROOT / 'train', transform=train_tf)
valid_ds = datasets.ImageFolder(DATA_ROOT / 'valid', transform=eval_tf)
test_ds = datasets.ImageFolder(DATA_ROOT / 'test', transform=eval_tf)
assert train_ds.classes == valid_ds.classes == test_ds.classes == EXPECTED_CLASSES
CLASSES = train_ds.classes
print(f'train={len(train_ds)}  valid={len(valid_ds)}  test={len(test_ds)}  classes={CLASSES}')

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## 4. Train `condition_mobilenetv2_4class_classifier_v1`
ImageNet-pretrained MobileNetV2, classifier head replaced (4 classes). Class-weighted loss
(inverse frequency) — PHYSICAL_DAMAGE is ~10x rarer than DUSTY in this split. Best-val-
accuracy checkpoint kept, early stopping on validation accuracy plateau.


In [ ]:
import torch.nn as nn

SEED = 42
EPOCHS = 30
PATIENCE = 6
LR = 5e-4
torch.manual_seed(SEED)
device = torch.device('cuda')

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASSES))
model.to(device)

counts = [0] * len(CLASSES)
for _, y in train_ds.samples:
    counts[y] += 1
class_weights = torch.tensor([len(train_ds) / (len(CLASSES) * max(1, c)) for c in counts],
                             dtype=torch.float32, device=device)
print('train class counts:', dict(zip(CLASSES, counts)))
print('loss weights       :', class_weights.tolist())

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

def run_epoch(loader, train):
    model.train(train)
    total_loss, total_correct, total_n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward(); optimizer.step()
            total_loss += float(loss.item()) * images.size(0)
            total_correct += int((logits.argmax(1) == labels).sum().item())
            total_n += images.size(0)
    return total_loss / total_n, total_correct / total_n

CKPT = Path('/kaggle/working/condition_mobilenetv2_4class_classifier_v1.pth')
history, best_val_acc, best_epoch, stale = [], -1.0, -1, 0
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    ep_t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(valid_loader, train=False)
    ep_dt = time.time() - ep_t0
    history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc,
                         val_loss=va_loss, val_acc=va_acc, seconds=ep_dt))
    print(f'epoch {epoch:2d}/{EPOCHS}  train_loss={tr_loss:.4f} train_acc={tr_acc:.3f}  '
          f'val_loss={va_loss:.4f} val_acc={va_acc:.3f}  ({ep_dt:.1f}s)')
    if va_acc > best_val_acc:
        best_val_acc, best_epoch, stale = va_acc, epoch, 0
        torch.save(model.state_dict(), CKPT)
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f'early stopping at epoch {epoch} — no val improvement for {PATIENCE} epochs '
                  f'(best epoch {best_epoch}, val_acc={best_val_acc:.3f})')
            break
train_seconds = time.time() - t0
print(f'\ntraining done in {train_seconds:.1f}s — best epoch {best_epoch}, val_acc={best_val_acc:.3f}')


## 5. Evaluate the BEST checkpoint on the held-out TEST split (evaluated once)


In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=device))
model.eval()
test_loss, test_acc = run_epoch(test_loader, train=False)

cm = [[0] * len(CLASSES) for _ in range(len(CLASSES))]
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images.to(device)).argmax(1).cpu()
        for t, p in zip(labels.tolist(), preds.tolist()):
            cm[t][p] += 1

per_class = {}
for i, cls in enumerate(CLASSES):
    tp = cm[i][i]; fn = sum(cm[i]) - tp; fp = sum(cm[r][i] for r in range(len(CLASSES))) - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else None
    recall = tp / (tp + fn) if (tp + fn) > 0 else None
    f1 = (2 * precision * recall / (precision + recall)) if precision and recall and (precision + recall) > 0 else None
    per_class[cls] = dict(n_test_images=sum(cm[i]), tp=tp, fp=fp, fn=fn,
                          precision=precision, recall=recall, f1=f1)

print(f'TEST accuracy: {test_acc:.3f}  (best epoch {best_epoch}, val_acc={best_val_acc:.3f})')
print('confusion matrix (rows=true, cols=pred, order=' + str(CLASSES) + '):')
for row in cm: print('  ', row)
for cls, m in per_class.items():
    print(f"  {cls:<16} n={m['n_test_images']:4d}  P={m['precision']}  R={m['recall']}  F1={m['f1']}")

BASELINE = dict(
    checkpoint='weights/yolo_solar_condition_4class_research_v2.pt',
    box_level_test_recall=0.441, image_level_agreement=0.727,
)
print('\nBaseline (box detector, image-level agreement):', BASELINE['image_level_agreement'])
print('This classifier (test accuracy)               :', test_acc)


## 6. Package artifacts + reproducibility manifest + SHA-256


In [ ]:
sha = hashlib.sha256(CKPT.read_bytes()).hexdigest()

manifest = dict(
    experiment_name='condition_mobilenetv2_4class_classifier_v1',
    phase='16-B — panel-crop condition classifier',
    model_role='EXPERIMENTAL RESEARCH condition classifier (second stage, on panel crops) — NOT a panel detector, NOT production, NOT validated for field use',
    architecture='torchvision MobileNetV2, ImageNet-pretrained, classifier head replaced (num_classes=4)',
    classes_index_order=CLASSES,
    baseline_for_comparison=BASELINE,
    dataset_kaggle='edithstark/solar-ai-ground-condition-4class-classifier-v1',
    dataset_derivation=report['label_derivation_rule'],
    minority_annotation_images_discarded=report['images_with_minority_annotation_discarded'],
    provenance_grade='C', provenance_tag='personal-experimental-unverified-provenance',
    train_class_counts=dict(zip(CLASSES, counts)),
    class_loss_weights=dict(zip(CLASSES, class_weights.tolist())),
    hyperparameters=dict(epochs_requested=EPOCHS, epochs_run=len(history), batch_size=BATCH_SIZE,
                         lr=LR, optimizer='AdamW', patience=PATIENCE, seed=SEED,
                         input_size=INPUT_SIZE, device=str(device)),
    best_epoch=best_epoch, best_val_accuracy=best_val_acc, training_seconds=train_seconds,
    history=history,
    test_set=dict(n_images=len(test_ds), overall_accuracy=test_acc, overall_loss=test_loss,
                  confusion_matrix=cm, confusion_matrix_class_order=CLASSES, per_class=per_class),
    checkpoint_sha256=sha, environment=ENV,
    not_production_validated=True,
)
OUT = Path('/kaggle/working/output'); OUT.mkdir(exist_ok=True)
shutil.copy2(CKPT, OUT / CKPT.name)
(OUT / 'training_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print('checkpoint sha256:', sha)
print(json.dumps(manifest, indent=2, default=str))
print('\nArtifacts:', sorted(p.name for p in OUT.iterdir()))


## 7. Retrieve
Download `/kaggle/working/output/`. Locally save the checkpoint as
`weights/mobilenet_condition_4class_research_v1.pth` (gitignored research artifact) —
**never** overwrite any existing production or research checkpoint.

This classifier is a research artifact only, NOT production-validated. It is a **second-
stage** model that runs on panel crops (the dynamic panel detector feeds it), exactly like
the box-detector condition model it's compared against — same input, different output
formulation (single label vs. localized boxes). Compare `test_set.overall_accuracy` here
against the 0.727 image-level-agreement figure from Phase 16-A.1 for the apples-to-apples
read; do NOT compare it directly against the box detector's 0.444 mAP50 (different metric
entirely — mAP is box-level, this is image-level).
